<!-- bootcamp-header: generated by tools/build_headers.py, edit the README timetable instead -->
# Input, output and files

**Session:** Monday 28 September 2026, 10:30-12:30, room S.SJ.117  
**Tutors:** Mike Kestemont / TBD  
**Exercises:** [`06_EX_Input_Output_Files.ipynb`](https://github.com/mikekestemont/dtaantwerp26-27.github.io/blob/DTA_Bootcamp_2026_students/exercises/questions/06_EX_Input_Output_Files.ipynb) (solutions: [`06_SOL_Input_Output_Files.ipynb`](https://github.com/mikekestemont/dtaantwerp26-27.github.io/blob/DTA_Bootcamp_2026_students/exercises/solutions/06_SOL_Input_Output_Files.ipynb))  

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mikekestemont/dtaantwerp26-27.github.io/blob/DTA_Bootcamp_2026_students/notebooks/06_W2_Mon_Input_Output_Files.ipynb)

In [ ]:
# Run this cell only if you are working on Google Colab: it downloads the course
# material (notebooks and data) so that the file paths in this notebook work.
# On your own computer you can skip it.
!git clone --quiet --depth 1 --branch DTA_Bootcamp_2026_students https://github.com/mikekestemont/dtaantwerp26-27.github.io.git bootcamp
%cd bootcamp/notebooks

## Input and output

A program is not an island: it takes data in and gives results out. So far, all the data your programs worked with was typed into the notebook itself. That does not scale: the texts you will analyse in this programme are files on a disk, often thousands of them, and the results of your analyses have to be saved somewhere, because everything in a notebook is gone when the kernel stops. This session is about **I/O** (input/output): getting data into a program and results out of it.

After this session you can:

- ask the user for a value with `input()`, and keep asking until the answer is valid;
- find out in which folder your notebook is running, and reach a file with a relative path;
- write text to a file, line by line, and add to an existing file;
- read a file into one string, into a list of lines, or line by line;
- open every file with `with` and `encoding='utf-8'`, and know what to do when a file will not decode.

## Keyboard input: `input()`

The simplest form of input is the keyboard. The function `input()` shows a prompt, waits until the user types something and presses Enter, and hands you what was typed. Run the cell, type your name and press Enter:

In [ ]:
name = input('What is your name? ')
print(f'Hello, {name}!')

Whatever the user types, `input()` gives it to you as a **string**, even if it looks like a number. Run the next cell and enter two numbers:

In [ ]:
a = input('Give a first number: ')
b = input('Give a second number: ')
print(type(a))
print('Sum:', a + b)

`3 + 4` is not `34`: the `+` concatenated two strings. When you want a number, **cast** the input with `int()` or `float()`:

In [ ]:
a = int(input('Give a first number: '))
b = int(input('Give a second number: '))
print('Sum:', a + b)

The user may also press Enter without typing anything. `input()` then returns the empty string `''`, which counts as `False` in a test (remember truthiness from the session on `if` and `else`), so this is easy to catch:

In [ ]:
name = input('What is your name? ')

if not name:
    print('You did not enter a name.')
else:
    print(f'Hello, {name}!')

### Keep asking until the answer is valid

A program that asks once and then crashes on a wrong answer is not very friendly. The standard solution is a `while True:` loop around the `input()`: if the answer is acceptable, `break`; otherwise, explain what is wrong and the loop asks again. Here the only acceptable answers are `yes` and `no`:

In [ ]:
while True:
    answer = input('Do you want to continue? (yes/no) ')
    if answer in ['yes', 'no']:
        break
    print('Please answer yes or no.')

print(f'You answered: {answer}')

Casting a wrong answer with `int()` would crash the program (`int('seven')` raises a `ValueError`), so test first. Strings have a method for exactly this: `'42'.isdigit()` is `True` if the string consists of digits only (and of nothing else), `'4.2'.isdigit()` and `'seven'.isdigit()` are `False`.

In [ ]:
while True:
    answer = input('Pick a number between 1 and 10: ')
    if answer.isdigit() and int(answer) >= 1 and int(answer) <= 10:
        number = int(answer)
        break
    print('That is not a whole number between 1 and 10.')

print(f'You picked {number}.')

Keep this pattern in mind; you will need it again in the project session. Note that the `and` test relies on something you saw in the session on `if` and `else`: Python evaluates `answer.isdigit()` first, and only if that is `True` does it try `int(answer)`, so the cast is safe.

### Class exercise: guess the number

Python comes with a large **standard library** of modules that you can load with `import`. The module `random` produces random values; after `import random`, the function `random.randint(1, 10)` returns a random whole number between 1 and 10 (both included). You will hear more about modules in the session on functions.

Write a guessing game: the computer picks a secret number between 1 and 10, the user has at most 5 guesses. After each wrong guess, tell the user whether the secret number is higher or lower than the guess. When the user guesses right, stop with a message of joy; when the guesses run out, reveal the number.

In [ ]:
import random

secret = random.randint(1, 10)

# your code here

## Files

Keyboard input is fine for small programs, but the data you will really work with lives in **files**: a text file is nothing more than a sequence of characters stored on a disk under a name. Reading files is how every corpus and every dataset gets into Python; writing files is how you keep your results after the kernel is gone.

### Where am I? Folders and paths

Before Python can open a file, it needs to know *where* the file is. Two questions, then. First: in which folder is my notebook running? The module `os` (for *operating system*) answers that with `os.getcwd()`, the **current working directory**:

In [ ]:
import os

print(os.getcwd())

When you start a notebook from Jupyter, the working directory is the folder that contains the notebook. All paths you write are understood **relative** to that folder:

- `'test.txt'` is a file in the working directory itself;
- `'../data/alice.txt'` goes one folder up (`..`), then into `data`, then to `alice.txt`. This is how the notebooks of this course reach the texts in the `data` folder, which sits next to the `notebooks` folder;
- `'/Users/alice/Documents/corpus.txt'` or `'C:/Users/alice/Documents/corpus.txt'` is an **absolute** path, which starts at the root of the disk. It works only on your own computer, so prefer relative paths in anything you share.

Use forward slashes, also on Windows; Python translates them. To see what a folder contains, use `os.listdir()`:

In [ ]:
print(os.listdir('../data'))

And `os.path.exists()` tells you whether a path leads somewhere. Compare:

In [ ]:
print(os.path.exists('../data/alice.txt'))
print(os.path.exists('../data/alice.text'))   # a typo in the name
print(os.path.exists('data/alice.txt'))       # there is no data folder *inside* the notebooks folder

Remember this when something "does not work on my laptop": a `FileNotFoundError` almost always means that the path is wrong for the folder you are in, not that the file is gone. Print `os.getcwd()` and `os.listdir()` and compare. One more trap: on Linux, and therefore on Colab and on most servers, file names are case sensitive (`Alice.txt` is not `alice.txt`), while Windows and macOS let you get away with the wrong case. Copy names exactly.

### Writing to a file

Let's create your first text file. To write to a file, you `open()` it, which gives you a **file object**: not the file itself, but an open connection to it, through which you can send text.

In [ ]:
outfile = open('test.txt', 'w', encoding='utf-8')
print(type(outfile))

`open()` gets three arguments here:

1. the **path** of the file: `test.txt`, so a file in the working directory (next to this notebook);
2. the **mode**: `'w'` for *write*. The file is created if it does not exist yet;
3. the **encoding**: always `'utf-8'`. One paragraph on why, because this will save you hours some day.

> A file stores bytes, not characters, and an **encoding** is the table that translates characters into bytes and back. There are many such tables, and the same bytes mean different characters in different tables. **UTF-8** is the one that can represent every character of every script, and it is the standard on the web and in this programme. If you do not name an encoding, Python uses whatever your operating system prefers, which on Windows is often *not* UTF-8, so the same file can read fine on one laptop and produce garbage or an error on the next. The rule of this course: **every `open()` gets `encoding='utf-8'`**. You will meet encodings in depth in the module *Text as data*.

Now we can send text to the file with `.write()`:

In [ ]:
outfile.write('Here goes some text.')

(The number is the number of characters written; you can ignore it.) One step remains, and it is important: **closing** the file. Until you close it, the text may still be sitting in a buffer instead of on the disk, and other programs cannot use the file properly.

In [ ]:
outfile.close()

After closing, the connection is gone and writing to it fails, with a clear message:

In [ ]:
# This cell produces an error on purpose: the file is closed
outfile.write('Some more text.')

Look at the folder of this notebook in your Finder or Explorer: `test.txt` is there. Open it in a text editor to check its contents. Or ask Python:

In [ ]:
print('test.txt' in os.listdir())   # no argument: the working directory

**Careful with `'w'`.** Opening a file in write mode erases everything it contained, immediately, without asking, and there is no Bin to recover it from:

In [ ]:
outfile = open('test.txt', 'w', encoding='utf-8')
outfile.close()

Open `test.txt` again in your editor: it is empty now. It is surprisingly easy to destroy a file this way. Check your paths and modes before you run a cell that writes.

### Lines and numbers

Two things that surprise everyone the first time. First, `.write()` writes exactly what you give it, and no more: unlike `print()`, it does not add a line break. Second, it accepts only strings. Numbers have to be converted first, for instance with an f-string.

In [ ]:
outfile = open('test.txt', 'w', encoding='utf-8')
outfile.write('This is the first line')
outfile.write('This is the second line')
outfile.close()

In [ ]:
# This cell produces an error on purpose: write() only accepts strings
outfile = open('test.txt', 'w', encoding='utf-8')
outfile.write(2026)
outfile.close()

Have a look at the file after the first cell: both lines ended up on one line. For a line break you add the special character `'\n'` (*newline*) yourself, exactly as in a `print()`:

In [ ]:
outfile = open('test.txt', 'w', encoding='utf-8')
outfile.write('This is the first line\n')
outfile.write('This is the second line\n')
outfile.write(f'This is line number {3}\n')
outfile.close()

In practice you rarely write lines by hand: you write them in a **loop**. Here we save two lists that belong together, as one line per novel. `'\t'` is a second special character, the **tab**; a file with one record per line and tabs between the fields is a *tab-separated values* file, which every spreadsheet program, and later `pandas`, can open directly.

In [ ]:
titles = ['Emma', 'Dracula', 'Middlemarch']
years = [1815, 1897, 1871]

outfile = open('novels.txt', 'w', encoding='utf-8')
for title, year in zip(titles, years):
    outfile.write(f'{title}\t{year}\n')
outfile.close()

### Adding to a file: append mode

Write mode starts from an empty file. To keep what is there and add to the end, open the file in **append** mode, `'a'`:

In [ ]:
outfile = open('novels.txt', 'a', encoding='utf-8')
outfile.write('Persuasion\t1817\n')
outfile.close()

Open `novels.txt` in your editor: four novels. Run the cell above once more and look again: append mode adds, every time you run it.

### Reading from a file

Reading works the same way, in mode `'r'` (which is the default, so you may leave it out): open, read, close. `.read()` gives you the whole file as **one string**:

In [ ]:
infile = open('novels.txt', 'r', encoding='utf-8')
text = infile.read()
infile.close()

print(type(text), len(text))
print(text)

The line breaks and tabs are still in there: we put them there ourselves. `print()` renders them; if you show the raw string instead, you see the `\n` and `\t` characters as such:

In [ ]:
text

More often than the text as one lump, you want its **lines**. There are three ways, and it is worth knowing all three.

**1. Split the string yourself.** `text.split('\n')` cuts at every newline. (Its close relative `text.splitlines()` does the same, but does not leave an empty string at the end when the file ends with a newline, as files usually do. Compare the two.)

In [ ]:
lines = text.split('\n')
print(lines)
print(len(lines))

In [ ]:
lines = text.splitlines()
print(lines)
print(len(lines))

**2. Let the file object do it.** `.readlines()` returns a list of lines straight away:

In [ ]:
infile = open('novels.txt', encoding='utf-8')
lines = infile.readlines()
infile.close()

print(lines)

Note the `'\n'` dangling at the end of every line: `.readlines()` keeps the line breaks. The string method `.strip()` removes whitespace (spaces, tabs, newlines) from both ends of a string, and you will use it constantly when reading files:

In [ ]:
for line in lines:
    print(line.strip())

**3. Loop over the file object.** File objects are **iterable**: a `for` loop over an open file walks through its lines, one at a time. This is the best way for large files, because the whole text is never in memory at once, only the current line. (It is also why you cannot ask an open file for its `len()`: it does not know yet how many lines are coming.)

In [ ]:
infile = open('novels.txt', encoding='utf-8')
for line in infile:
    print(line)
infile.close()

Why the empty lines? Each `line` ends in a `'\n'`, and `print()` adds a second one. Either `print(line.strip())` or `print(line, end='')` fixes it.

One more thing to know about file objects: they remember **how far you have read**. A second `.read()` on the same open file returns nothing, because the first one read up to the end:

In [ ]:
infile = open('novels.txt', encoding='utf-8')
first = infile.read()
second = infile.read()
infile.close()

print(len(first), len(second))

The cure is simple: open the file again when you need its contents a second time, or, better, read it once into a variable and work with the variable.

### `with`: never forget to close a file again

It is easy to forget a `.close()`, especially when something goes wrong halfway. Python has a construction that closes the file for you: `with`. The syntax is `with open(...) as` a variable name of your choice, a colon, and an indented block. Inside the block the file is open; the moment the block ends, whether normally or through an error, the file is closed.

In [ ]:
with open('novels.txt', encoding='utf-8') as infile:
    text = infile.read()

print(text)

Two things to notice. The variable `text` is still there after the block: `with` closes the *file*, not your variables. And the file really is closed:

In [ ]:
print(infile.closed)

`with` works for writing and appending too:

In [ ]:
with open('novels.txt', 'a', encoding='utf-8') as outfile:
    outfile.write('Kidnapped\t1886\n')

From here on, **every file in this course is opened with `with`** (and with `encoding='utf-8'`). We showed `open()` and `close()` separately so that you understand what `with` does for you; you should not write them yourself any more.

### A real text: *Alice in Wonderland*

Time for a real file. `data/alice.txt` is the Project Gutenberg text of *Alice's Adventures in Wonderland*. Reading it is no different from reading `novels.txt`:

In [ ]:
with open('../data/alice.txt', encoding='utf-8') as infile:
    alice = infile.read()

print(len(alice))
print(alice[:500])

### Class exercises

Everything you need is above, plus what you know from the sessions on strings, lists, `if` and loops.

1. How many lines does `alice.txt` have, and how many of them are empty? (A line is empty when nothing is left of it after `.strip()`.)

In [ ]:
# your code here

2. How often does the name `Alice` occur in the text? And how often the word `said`, ignoring case?

In [ ]:
# your code here

3. Print the first ten lines that are not empty, without their line breaks.

In [ ]:
# your code here

4. Write every line in which the Queen appears (the line contains `Queen`) to a new file `queen.txt`, one line per line, and print how many lines you wrote. Open the file in your editor to check.

In [ ]:
# your code here

### When a file will not decode

Sooner or later you will open a file and get a `UnicodeDecodeError`. It means what the paragraph on encodings said: the file was saved with a different table than the one you are reading it with. Let's provoke one. We write a file in `latin-1`, an older encoding that is still common for Western European text, and read it as UTF-8:

In [ ]:
with open('test.txt', 'w', encoding='latin-1') as outfile:
    outfile.write('Un café, s\'il vous plaît.')

In [ ]:
# This cell produces an error on purpose: the file is not UTF-8
with open('test.txt', encoding='utf-8') as infile:
    print(infile.read())

The message names the byte that could not be decoded (`0xe9`, the `é` in `latin-1`). The fix is to name the right encoding, which you often have to guess: `'latin-1'` and `'cp1252'` (the Windows variant) are the usual suspects for European text.

In [ ]:
with open('test.txt', encoding='latin-1') as infile:
    print(infile.read())

A milder variant of the same problem: some UTF-8 files start with an invisible marker, the **byte order mark** (BOM), that certain Windows programs and Project Gutenberg add. The file decodes fine, but its first character is a strange `'\ufeff'` that then confuses your string comparisons and counts. The Sherlock Holmes text in `data` is such a file:

In [ ]:
with open('../data/Sherlock.txt', encoding='utf-8') as infile:
    sherlock = infile.read()

print(repr(sherlock[:30]))   # repr() shows the string the way Python sees it, invisible characters included

The encoding `'utf-8-sig'` reads UTF-8 and drops the marker if there is one. It is safe to use on any UTF-8 file, so when in doubt, use it:

In [ ]:
with open('../data/Sherlock.txt', encoding='utf-8-sig') as infile:
    sherlock = infile.read()

print(repr(sherlock[:30]))

## Optional: saving Python objects with `json`

A text file stores text. If you save a list of titles with `.write()`, you save characters, and when you read the file back you get one string that you have to split again. When you want to save a Python object *and get the same object back*, use the `json` module. **JSON** is a small text format, readable by humans and by almost every programming language, that can hold strings, numbers, booleans, `None`, lists, and (from the next session on) dictionaries. `json.dump()` writes an object to an open file; `json.load()` reads it back:

In [ ]:
import json

novels = [['Emma', 1815], ['Dracula', 1897], ['Middlemarch', 1871]]

with open('novels.json', 'w', encoding='utf-8') as outfile:
    json.dump(novels, outfile)

In [ ]:
with open('novels.json', encoding='utf-8') as infile:
    novels_again = json.load(infile)

print(novels_again)
print(type(novels_again), type(novels_again[0]), type(novels_again[0][1]))
print(novels_again == novels)

Have a look at `novels.json` in your editor: it looks almost exactly like the Python list. You will use JSON constantly later in the programme, for corpora, annotations and results alike.

## Common mistakes

Three errors you will meet this week. Read the messages: they tell you what is wrong.

In [ ]:
# This cell produces an error on purpose: the path is wrong for this folder (the data folder is one level up)
with open('data/alice.txt', encoding='utf-8') as infile:
    alice = infile.read()

In [ ]:
# This cell produces an error on purpose: the file is open for reading, not for writing
with open('novels.txt', encoding='utf-8') as infile:
    infile.write('Dracula\t1897\n')

In [ ]:
# This cell produces an error on purpose: the file object is not a string
with open('novels.txt', encoding='utf-8') as infile:
    print(infile.count('Emma'))

The third one is a classic: the file object is the *connection*, not the text. Call `.read()` first, and then use string methods on the result.

## References

- [Reading and writing files](https://docs.python.org/3/tutorial/inputoutput.html#reading-and-writing-files) in the official Python tutorial
- [`os`](https://docs.python.org/3/library/os.html) and [`json`](https://docs.python.org/3/library/json.html) in the Python documentation
- [Unicode HOWTO](https://docs.python.org/3/howto/unicode.html), for when you want to understand encodings properly